In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip -q install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 942.1 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [3]:
from transformers import AutoModelForTokenClassification, AutoTokenizer,DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
import torch
import evaluate  # pip install evaluate
import seqeval   # pip install seqeval
from datasets import load_dataset

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
import torch.optim as optim

In [4]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-chinese')

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# 加载hf中dataset
ds = load_dataset('doushabao4766/msra_ner_k_V3')
ds

README.md:   0%|          | 0.00/697 [00:00<?, ?B/s]

data/train-00000-of-00001-42717a92413393(…):   0%|          | 0.00/13.9M [00:00<?, ?B/s]

data/test-00000-of-00001-8899cab5fdab45b(…):   0%|          | 0.00/946k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45001 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3443 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge'],
        num_rows: 45001
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge'],
        num_rows: 3443
    })
})

In [6]:
def tokenize_fn(examples):
    result = tokenizer(
        examples['tokens'], 
        max_length=512, 
        truncation=True, 
        add_special_tokens=False,
        is_split_into_words=True
    )
    
    labels = []
    for i, ner_tags in enumerate(examples['ner_tags']):
        word_ids = result.word_ids(batch_index=i)
        labels.append([ner_tags[w] for w in word_ids])
    
    result['labels'] = labels
    return result

ds1 = ds.map(tokenize_fn, batched=True)


Map:   0%|          | 0/45001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3443 [00:00<?, ? examples/s]

In [7]:
ds1.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [8]:
ds1['train'].features

{'id': Value('string'),
 'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])),
 'knowledge': Value('string'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8')),
 'labels': List(Value('int64'))}

In [9]:
for item in ds1['train']:
    print(item['labels'])
    print(item['input_ids'])
    break

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0])
tensor([2496, 2361, 3307, 2339, 4923, 3131, 1221, 4638, 4636,  674, 1036, 4997,
        2768, 7270, 6629, 3341, 8024, 4906, 3136, 1069, 1744, 5917, 4197, 2768,
        7599, 3198, 8024,  791, 1921, 3300, 3119, 5966,  817,  966, 4638,  741,
         872, 3766,  743, 8024, 3209, 3189, 2218, 1373,  872, 2637,  679, 2496,
        1159, 8013])


In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,           # 必需：用于获取 pad_token_id
    padding=True,                  # 默认 True，动态 padding
    max_length=None,               # 可选：最大长度限制
    return_tensors="pt"            # 返回 PyTorch 张量
)


train_dl = DataLoader(ds1['train'], shuffle=True, batch_size=16, collate_fn=data_collator)


# 模型创建
tags = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
id2lbl = {i:tag for i, tag in enumerate(tags)}
lbl2id = {tag:i for i, tag in enumerate(tags)}


model = AutoModelForTokenClassification.from_pretrained('google-bert/bert-base-chinese', 
                                                        num_labels=7, 
                                                        id2label=id2lbl, 
                                                        label2id=lbl2id)
model.to(device)

# 模型参数分组
param_optimizer = list(model.named_parameters())
bert_params, classifier_params = [],[]

for name,params in param_optimizer:
    if 'bert' in name:
        bert_params.append(params)
    else:
        classifier_params.append(params)

param_groups = [
    {'params':bert_params, 'lr':1e-5},
    {'params':classifier_params, 'weight_decay':0.1, 'lr':1e-3}
]


# optimizer
optimizer = optim.AdamW(param_groups) # 优化器

# 学习率调度器
train_steps = len(train_dl) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=100, 
                                            num_training_steps=train_steps)


model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

In [11]:
for item in train_dl:
    print(item['input_ids'].shape, 
          item['token_type_ids'].shape, 
          item['attention_mask'].shape, 
          item['labels'].shape)
    break

torch.Size([16, 57]) torch.Size([16, 57]) torch.Size([16, 57]) torch.Size([16, 57])


In [13]:
from tqdm import tqdm


for epoch in range(5):
    model.train()
    tpbar = tqdm(train_dl)
    for items in tpbar:
        items = {k:v.to(device) for k,v in items.items()}
        optimizer.zero_grad()
        outputs = model(**items)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
    
        tpbar.set_description(f'Epoch:{epoch+1} ' + 
                          f'bert_lr:{scheduler.get_lr()[0]} ' + 
                          f'classifier_lr:{scheduler.get_lr()[1]} '+
                          f'Loss:{loss.item():.4f}')

Epoch:1 bert_lr:9.947010383100609e-06 classifier_lr:0.0009947010383100608 Loss:0.1752:   0%|          | 4/2813 [00:01<15:37,  3.00it/s]


KeyboardInterrupt: 

知道learning rate在缓慢变化就好了

## 支持混合精度训练

In [14]:
train_dl = DataLoader(ds1['train'], shuffle=True, batch_size=16, collate_fn=data_collator)


# 模型创建
tags = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
id2lbl = {i:tag for i, tag in enumerate(tags)}
lbl2id = {tag:i for i, tag in enumerate(tags)}


model = AutoModelForTokenClassification.from_pretrained('google-bert/bert-base-chinese', 
                                                        num_labels=7, 
                                                        id2label=id2lbl, 
                                                        label2id=lbl2id)
model.to(device)

# 模型参数分组
param_optimizer = list(model.named_parameters())
bert_params, classifier_params = [],[]

for name,params in param_optimizer:
    if 'bert' in name:
        bert_params.append(params)
    else:
        classifier_params.append(params)

param_groups = [
    {'params':bert_params, 'lr':1e-5},
    {'params':classifier_params, 'weight_decay':0.1, 'lr':1e-3}
]


# optimizer
optimizer = optim.AdamW(param_groups) # 优化器

# 学习率调度器
train_steps = len(train_dl) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=100, 
                                            num_training_steps=train_steps)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

In [15]:
# 梯度计算缩放器
scaler = torch.GradScaler()

for epoch in range(5):
    model.train()
    tpbar = tqdm(train_dl)
    for items in tpbar:
        items = {k:v.to(device) for k,v in items.items()}
        optimizer.zero_grad()

        with torch.autocast(device_type=device):
            outputs = model(**items)
        loss = outputs.loss

        # 缩放loss后，调用backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
    
        tpbar.set_description(f'Epoch:{epoch+1} ' + 
                          f'bert_lr:{scheduler.get_lr()[0]} ' + 
                          f'classifier_lr:{scheduler.get_lr()[1]} '+
                          f'Loss:{loss.item():.4f}')

Epoch:1 bert_lr:8.057286072323668e-06 classifier_lr:0.0008057286072323666 Loss:0.0195: 100%|██████████| 2813/2813 [05:54<00:00,  7.94it/s]
Epoch:2 bert_lr:7.828141783029003e-06 classifier_lr:0.0007828141783029002 Loss:0.0041:  11%|█▏        | 320/2813 [00:39<05:09,  8.06it/s]


KeyboardInterrupt: 

In [35]:
%%writefile ddp_simple.py

import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler


# 设置分布式环境
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

# 清理分布式环境
def cleanup():
    dist.destroy_process_group()


# 保存模型
def save_model(model, optimizer, epoch, save_dir='./checkpoints'):
    """保存模型检查点"""
    os.makedirs(save_dir, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.module.state_dict(),  # DDP模型需要.module获取原始模型
        'optimizer_state_dict': optimizer.state_dict(),
    }
    save_path = os.path.join(save_dir, f'model_epoch_{epoch}.pth')
    torch.save(checkpoint, save_path)
    print(f"模型已保存到 {save_path}")


# 预测功能
def predict(model, image_tensor, device='cuda', class_names=None):
    """对单张图像进行预测

    Args:
        model: 加载好的模型
        image_tensor: 预处理后的图像张量 (C, H, W) 或 (1, C, H, W)
        device: 设备
        class_names: 类别名称列表，如不提供则返回索引

    Returns:
        预测的类别名称或索引
    """
    model.eval()
    with torch.no_grad():
        if image_tensor.dim() == 3:
            image_tensor = image_tensor.unsqueeze(0)
        image_tensor = image_tensor.to(device)

        with torch.autocast(device_type='cuda'):
            output = model(image_tensor)
            _, predicted = torch.max(output, 1)

    pred_idx = predicted.item()
    if class_names:
        return class_names[pred_idx]
    return pred_idx


# 定义训练循环
def train(rank, world_size):
    setup(rank, world_size)

    # 定义模型并将其移动到对应的 GPU 设备端
    model = models.resnet50()
    model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 有10个类别
    model = model.to(rank)
    ddp_model = DDP(model, device_ids=[rank])

    # 损失函数及优化器
    criterion = nn.CrossEntropyLoss().to(rank)
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.01)

    # 定义数据集Dataset的转换和图像增强
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    # 分布式训练采样器
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)

    # 在训练开始时创建一次 梯度计算缩放器
    scaler = torch.amp.GradScaler(device=rank)

    for epoch in range(10):
        # 每个epoch开始时设置epoch，确保每个epoch的数据shuffle不同
        sampler.set_epoch(epoch)

        ddp_model.train()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(rank), labels.to(rank)
            optimizer.zero_grad()

            with torch.autocast(device_type='cuda'):
                outputs = ddp_model(inputs)
                loss = criterion(outputs, labels)

            # 缩放loss后调用backward
            scaler.scale(loss).backward()

            # 先取消梯度缩放，再进行梯度裁剪
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1)

            scaler.step(optimizer)
            scaler.update()

            if rank == 0:  # 只在主进程打印
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

        # 只在主进程保存模型
        if rank == 0:
            save_model(ddp_model, optimizer, epoch, save_dir='./checkpoints')

    cleanup()


def main():
    world_size = torch.cuda.device_count()
    if world_size < 2:
        print(f"警告: 检测到 {world_size} 个GPU，分布式训练建议使用至少2个GPU")
    mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)


if __name__ == "__main__":
    main()




Overwriting ddp_simple.py


In [36]:
!python ddp_simple.py

Epoch 0, Loss: 2.4410
Epoch 0, Loss: 2.3978
Epoch 0, Loss: 2.5028
Epoch 0, Loss: 2.4057
Epoch 0, Loss: 2.3440
Epoch 0, Loss: 2.4005
Epoch 0, Loss: 2.5085
Epoch 0, Loss: 2.4321
Epoch 0, Loss: 2.3363
Epoch 0, Loss: 2.4065
Epoch 0, Loss: 2.3734
Epoch 0, Loss: 2.5626
Epoch 0, Loss: 2.4717
Epoch 0, Loss: 2.3881
Epoch 0, Loss: 2.5372
Epoch 0, Loss: 2.2772
Epoch 0, Loss: 2.4129
Epoch 0, Loss: 2.3005
Epoch 0, Loss: 2.3395
Epoch 0, Loss: 2.3633
Epoch 0, Loss: 2.4015
Epoch 0, Loss: 2.3845
Epoch 0, Loss: 2.3604
Epoch 0, Loss: 2.4534
Epoch 0, Loss: 2.4097
Epoch 0, Loss: 2.3582
Epoch 0, Loss: 2.4375
Epoch 0, Loss: 2.4803
Epoch 0, Loss: 2.3378
Epoch 0, Loss: 2.3292
Epoch 0, Loss: 2.4318
Epoch 0, Loss: 2.3588
Epoch 0, Loss: 2.3218
Epoch 0, Loss: 2.4080
Epoch 0, Loss: 2.4154
Epoch 0, Loss: 2.3080
Epoch 0, Loss: 2.3522
Epoch 0, Loss: 2.3330
Epoch 0, Loss: 2.2469
Epoch 0, Loss: 2.3474
Epoch 0, Loss: 2.3816
Epoch 0, Loss: 2.3402
Epoch 0, Loss: 2.3270
Epoch 0, Loss: 2.3516
Epoch 0, Loss: 2.3542
Epoch 0, L

In [37]:
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

In [38]:

# 加载模型
def load_model(model, checkpoint_path, device='cuda'):
    """加载模型检查点"""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"模型已从 {checkpoint_path} 加载")
    return model, checkpoint.get('epoch', -1), checkpoint.get('optimizer_state_dict', None)


# 推理预测示例函数
def inference_demo(checkpoint_path):
    """加载模型并在测试集上评估准确率

    Args:
        checkpoint_path: 模型检查点路径
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 创建模型结构（与训练时一致，fc输出10类）
    model = models.resnet50()
    model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 有10个类别
    model, epoch, _ = load_model(model, checkpoint_path, device)
    model = model.to(device)
    print(f"加载的是第 {epoch} 个 epoch 的模型")

    # 定义预处理
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # 加载测试集
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # 评估准确率
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            with torch.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu'):
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"测试集准确率: {accuracy:.2f}% ({correct}/{total})")
    return accuracy



In [39]:
inference_demo('./checkpoints/model_epoch_1.pth')

模型已从 ./checkpoints/model_epoch_1.pth 加载
加载的是第 1 个 epoch 的模型
测试集准确率: 16.77% (1677/10000)


16.77

In [1]:
%%writefile ddp_trainer.py


import os
import numpy as np
from transformers import AutoModelForTokenClassification, AutoTokenizer,DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
import torch
import evaluate  # pip install evaluate
import seqeval   # pip install seqeval
from datasets import load_dataset
import torch.distributed as dist
import torch.multiprocessing as mp

# 设置分布式环境
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

# 清理分布式环境
def cleanup():
    dist.destroy_process_group()
    

def train(rank, world_size):
    setup(rank, world_size)
    # 数据集
    ds = load_dataset('nlhappy/CLUE-NER')
    # entity_index
    entites = ['O'] + ['movie', 'name', 'game', 'address', 'position', 'company', 'scene', 'book', 'organization', 'government']
    tags = ['O']
    for entity in entites[1:]:
        tags.append('B-' + entity.upper())
        tags.append('I-' + entity.upper())
    
    entity_index = {entity:i for i, entity in enumerate(entites)}

    tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-chinese')
    
    def entity_tags_proc(item):
        # item即是dataset中记录
        text_len = len(item['text'])  # 根据文本长度生成tags列表
        tags = [0] * text_len    # 初始值为‘O’
        # 遍历实体列表，所有实体类别标记填入tags
        entites = item['ents']
        for ent in entites:
            indices = ent['indices']  # 实体索引
            label = ent['label']   # 实体名
            tags[indices[0]] = entity_index[label] * 2 - 1
            for idx in indices[1:]:
                tags[idx] = entity_index[label] * 2
        return {'ent_tag': tags}
    
    # 使用自定义回调函数处理数据集记录
    ds1 = ds.map(entity_tags_proc)
    
    def data_input_proc(item):
        # 输入文本先拆分为字符，再转换为模型输入的token索引
        batch_texts = [list(text) for text in item['text']]
        # 导入拆分为字符的文本列表时，需要设置参数is_split_into_words=True
        input_data = tokenizer(batch_texts, truncation=True, add_special_tokens=False, max_length=512, 
                               is_split_into_words=True, padding='max_length')
        input_data['labels'] = [tag + [0] * (512 - len(tag)) for tag in item['ent_tag']]
        return input_data
        
    
    ds2 = ds1.map(data_input_proc, batched=True)  # batch_size 1000
    
    
    local_rank = rank
    
    id2lbl = {i:tag for i, tag in enumerate(tags)}
    lbl2id = {tag:i for i, tag in enumerate(tags)}
    
    model = AutoModelForTokenClassification.from_pretrained('google-bert/bert-base-chinese', 
                                                            num_labels=21,
                                                            id2label=id2lbl,
                                                            label2id=lbl2id)
    model.to(local_rank)
    
    args = TrainingArguments(
        output_dir="ner_train",  # 模型训练工作目录（tensorboard，临时模型存盘文件，日志）
        num_train_epochs = 3,    # 训练 epoch
        per_device_train_batch_size=16,  # 训练批次
        per_device_eval_batch_size=16,
        report_to='tensorboard',  # 训练输出记录
        eval_strategy="epoch",
        local_rank=local_rank,   # 当前进程 RANK
        fp16=True,               # 使用混合精度
        lr_scheduler_type='linear',  # 动态学习率
        warmup_steps=100,        # 预热步数
        ddp_find_unused_parameters=False  # 优化DDP性能
    )
    
    def compute_metric(result):
        # result 是一个tuple (predicts, labels)
        
        # 获取评估对象
        seqeval = evaluate.load('seqeval')
        predicts,labels = result
        predicts = np.argmax(predicts, axis=2)
        
        # 准备评估数据
        predicts = [[tags[p] for p,l in zip(ps,ls) if l != -100]
                     for ps,ls in zip(predicts,labels)]
        labels = [[tags[l] for p,l in zip(ps,ls) if l != -100]
                     for ps,ls in zip(predicts,labels)]
        results = seqeval.compute(predictions=predicts, references=labels)
    
        return results
    
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)
    
    trainer = Trainer(
        model,
        args,
        train_dataset=ds2['train'],
        eval_dataset=ds2['validation'],
        data_collator=data_collator,
        compute_metrics=compute_metric
    )
    
    trainer.train()

    cleanup()

def main():
    world_size = torch.cuda.device_count()
    mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)

if __name__ == "__main__":
    main()

Writing ddp_trainer.py


In [3]:
! pip install seqeval evaluate 


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=489b2bffcd69314715768dc32d68ecd9259040e9f563344b299fe3c28cd5186e
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
! python ddp_trainer.py

README.md: 100%|█████████████████████████████| 21.0/21.0 [00:00<00:00, 68.5kB/s]
dataset_infos.json: 100%|██████████████████████| 970/970 [00:00<00:00, 3.91MB/s]
data/train-00000-of-00001-a33d0e4276aef9(…): 100%|█| 1.30M/1.30M [00:00<00:00, 2
data/validation-00000-of-00001-07f476b71(…): 100%|█| 178k/178k [00:00<00:00, 1.6
Generating train split: 100%|██| 10748/10748 [00:00<00:00, 211757.31 examples/s]
Generating validation split: 100%|█| 1343/1343 [00:00<00:00, 341039.55 examples/
config.json: 100%|█████████████████████████████| 624/624 [00:00<00:00, 4.21MB/s]
tokenizer_config.json: 100%|██████████████████| 49.0/49.0 [00:00<00:00, 339kB/s]
vocab.txt: 110kB [00:00, 10.3MB/s]
tokenizer.json: 269kB [00:00, 16.5MB/s]
Map: 100%|█████████████████████████| 1343/1343 [00:00<00:00, 2042.44 examples/s]
model.safetensors: 100%|██████████████████████| 412M/412M [00:01<00:00, 256MB/s]
Loading weights: 100%|█| 197/197 [00:00<00:00, 2037.50it/s, Materializing param=
Loading weights: 100%|█| 197/197 [